# Запуск сервиса на Kaggle (показ на защите)

Ноутбук скачивает проект с GitHub, загружает модели на видеокарту, запускает веб-сервис и выдаёт
**публичную ссылку** — её можно открыть с любого устройства, пока ноутбук работает.

**Перед запуском** (панель справа, раздел *Session options*):
1. **Accelerator → GPU T4 x2.**
2. **Internet → On.** Нужен, чтобы скачать проект и модели.

Затем **Run All** (или запускайте ячейки по очереди). Через 5–8 минут в шаге 4 появится ссылка.

Пока идёт показ, **не закрывайте вкладку** с ноутбуком: сессия Kaggle работает, пока она открыта
(не дольше 12 часов). После показа остановите сессию кнопкой ⏻ — иначе тратится недельная квота
видеокарты (на бесплатном аккаунте — несколько часов в неделю, один запуск занимает 5–10 минут плюс время показа).

## 1. Проверка видеокарты

In [ ]:
import sys

import torch

print("Python", sys.version.split()[0], "| PyTorch", torch.__version__)
assert torch.cuda.is_available(), (
    "Видеокарта не включена: справа Session options → Accelerator → GPU T4 x2, затем запустите ноутбук заново")
gpu = torch.cuda.get_device_properties(0)
print(f"Видеокарта: {gpu.name}, {gpu.total_memory / 2**30:.0f} ГБ памяти")
if gpu.major < 7:
    print("⚠ Эта видеокарта слишком старая для PyTorch — выберите GPU T4 x2")

## 2. Проект и зависимости (~2 минуты)

In [ ]:
import os
from pathlib import Path

REPO = "https://github.com/nonqr/exit-interview-topics.git"
BRANCH = "main"
# Проект кладём во временную папку, а не в /kaggle/working: всё, что там лежит, Kaggle сохраняет как результат.
PROJECT = Path("/tmp/exit-interview-topics")

!rm -rf {PROJECT} && git clone -q --depth 1 --branch {BRANCH} {REPO} {PROJECT}
os.chdir(PROJECT)

# PyTorch, numpy и scipy берём из Kaggle: они уже собраны под его Python и видеокарту.
# Остальное ставим ровно тех версий, с которыми проект проверен.
reqs = [line for line in open("requirements-app.txt")
        if line.strip() and not line.startswith(("#", "numpy", "scipy", "torch"))]
Path("/tmp/requirements-demo.txt").write_text("".join(reqs))
%pip install -q -r /tmp/requirements-demo.txt
!git log -1 --format='Версия проекта: %h %s'

## 3. Запуск сервиса (~3–5 минут)

Модели (Qwen3-Embedding-4B, ~8 ГБ, и e5-large-instruct) скачиваются и загружаются на видеокарту
**до** старта сервиса, поэтому на показе загруженный файл считается сразу. Сразу после загрузки
векторы сверяются с эталонными (посчитанными заранее): косинус должен быть близок к 1.
Демо-датасет посчитан заранее и лежит в репозитории.

In [ ]:
import subprocess
import time
import urllib.request

SERVE_LOG = Path("/tmp/serve.log")


def server_ready() -> bool:
    try:
        return urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2).status == 200
    except OSError:
        return False


if not server_ready():
    server = subprocess.Popen([sys.executable, "-m", "src.app.serve"], cwd=PROJECT,
                              stdout=SERVE_LOG.open("w"), stderr=subprocess.STDOUT)
    shown = 0
    for _ in range(20 * 60):  # не дольше 20 минут
        lines = SERVE_LOG.read_text(errors="replace").splitlines()
        for line in lines[shown:]:
            if line.startswith(("Устройство", "Модель", "Запускаю", "Traceback", "RuntimeError")):
                print(line)
        shown = len(lines)
        if server_ready():
            break
        if server.poll() is not None:
            print(SERVE_LOG.read_text(errors="replace")[-3000:])
            raise RuntimeError("Сервис не запустился — лог выше")
        time.sleep(1)
print("Сервис работает" if server_ready() else "Сервис не ответил за 20 минут — смотрите лог в шаге 5")

## 4. Публичная ссылка

Туннель Cloudflare (бесплатно, без регистрации) делает сервис доступным снаружи. Ссылка начинает
открываться через 10–20 секунд после появления. Если она перестала работать, запустите эту ячейку
ещё раз — появится новая.

In [ ]:
import re

from IPython.display import HTML, display

CLOUDFLARED = Path("/tmp/cloudflared")
TUNNEL_LOG = Path("/tmp/tunnel.log")
if not CLOUDFLARED.exists():
    !wget -q -O {CLOUDFLARED} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    CLOUDFLARED.chmod(0o755)

if "tunnel" in globals() and tunnel.poll() is None:
    tunnel.terminate()
tunnel = subprocess.Popen([str(CLOUDFLARED), "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                          stdout=TUNNEL_LOG.open("w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    found = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", TUNNEL_LOG.read_text(errors="replace"))
    if found:
        url = found.group(0)
        break
    time.sleep(1)

if not url:
    print(TUNNEL_LOG.read_text(errors="replace")[-2000:])
    raise RuntimeError("Туннель не поднялся — проверьте, что включён Internet, и запустите ячейку ещё раз")

# Ссылка начинает открываться не сразу: проверяем её так же, как её откроет браузер.
opened = False
for _ in range(30):
    try:
        opened = urllib.request.urlopen(f"{url}/_stcore/health", timeout=5).status == 200
    except OSError:
        pass
    if opened:
        break
    time.sleep(2)
display(HTML(f'<h2>Сервис: <a href="{url}" target="_blank">{url}</a></h2>'))
print("Ссылка проверена: открывается" if opened else "Ссылка пока не открывается снаружи — подождите минуту и обновите страницу")

## 5. Если что-то не так: последние строки лога сервиса

In [ ]:
print(SERVE_LOG.read_text(errors="replace")[-3000:])

## 6. Остановить сервис (после показа)

In [ ]:
for proc in ("tunnel", "server"):
    if proc in globals() and globals()[proc].poll() is None:
        globals()[proc].terminate()
print("Остановлено. Не забудьте остановить сессию Kaggle кнопкой ⏻.")